# 3. Four-month windows and usable training pixels

Read saved products from [Sen4AgriNet preparation](Sen4AgriNet_Satellite_Preparation.ipynb) and [FranceCrops preparation](FranceCrops_Satellite_Preparation.ipynb). Audit each source in its own year (2020 and 2022), select supported windows, and export common features for [Model training](Crop_Model_Training.ipynb).

**Run sequentially in a fresh kernel.** No NetCDF or satellite downloads occur here. Missing actual-date summaries leave timing provisional.

**Pilot budget:** 3,000 candidate pixels per crop in the source notebooks; retain up to **1,000 usable unique pixels per crop** here, across both window policies. Save every complete window before trimming, report any shortfall, and preserve local evaluation rows. Original Steps 6–14 retain their numbers.


## Start here: load source handoffs

Output: `Source not prepared: <path>` — a source notebook has not exported its acquisition paths yet; available sources can still be inspected.

Output: `Prepared sources: <names>` — datasets actually loaded. All three crops are required for a complete three-class pilot.

Output: `Analysis folder: <path>` — separate expanded-pilot results, preserving the original run.


In [ ]:
from pathlib import Path
import json
import os
import importlib
from collections import Counter
import pilot_sources

settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).resolve()
source_handoff_folder = report_folder.parent / "source_handoffs"
source_manifest_paths = [
    source_handoff_folder / "sen4agrinet_3000.json",
    source_handoff_folder / "francecrops_3000.json",
]
sources = pilot_sources.load_sources(source_manifest_paths)
analysis_folder = report_folder.parent / "analysis" / "pilot_3000"
minimum_valid_fraction = 0.8
print("Prepared sources:", [source["source_name"] for source in sources])
print("Analysis folder:", analysis_folder)


## 6. Check quality, scaling and alignment

Audit every interval, including missing or failed downloads. Verify the combined-cloud-mask version, band order, sampling-mask grid and clear-observation counts. Scale spectral values by 0.0001 after handling nodata; never scale counts.

The initial coverage rule requires valid values at 80% of selected pixels. This is an exploratory setting to inspect before freezing. Preserve low-coverage and invalid records instead of filling missing values.

Original GeoTIFFs stay unchanged. On another PC, the helpers try current project and imagery roots before old absolute paths. Cloud-contaminated medians require upstream regeneration; quality checks cannot repair them afterwards.


### 6.1 Audit all pilot intervals

Run local quality checks using the previous settings and existing GeoTIFFs. Missing intervals remain in the report.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `<region ID> quality audit completed` | The region was processed by the audit. This message alone does not mean all of its bins passed. |
| `Quality status: {'quality_pass': <number>, 'low_coverage': <number>, ...}` | Counts region/bin records, not individual pixels or parcels. quality_pass meets the coverage rule; low_coverage has insufficient valid selected pixels; missing/invalid records identify unavailable or incompatible imagery. |
| `Reports: <path>` | The folder containing per-bin quality, scaled selected-pixel bands and errors. These records feed timelines and features. |
<!-- output-guide:end -->


In [ ]:
import crop_workflow

importlib.reload(crop_workflow)
regions, quality_rows, pixel_rows = pilot_sources.audit_sources(
    sources, analysis_folder, minimum_valid_fraction
)
# Combined tables let provenance packaging record both source years explicitly.
quality_folder = analysis_folder / "quality"
crop_workflow.write_table(
    quality_folder / "composite_quality.csv",
    quality_rows,
    list(quality_rows[0]) if quality_rows else [],
)
crop_workflow.write_table(
    quality_folder / "selected_pixel_bands.csv",
    pixel_rows,
    list(pixel_rows[0]) if pixel_rows else [],
)
print("Quality status:", dict(Counter(row["status"] for row in quality_rows)))
print("Reports:", quality_folder)


### Outputs and interpretation

`analysis/2020/quality/` contains quality records, scaled selected-pixel bands, settings and errors. Missing and low-coverage intervals remain explicit.


## 7. Build vegetation timelines

Calculate NDVI, NDRE and NDMI from each selected pixel's composite bands. Then summarize the selected pixels in each region. Composite-bin midpoints are display dates, not actual acquisition dates.

For stronger onset evidence, retrieve small daily index summaries using the same cloud mask, sampling footprint and same-date mosaic rule. This transfers tables only, not additional image files. Checkpoints let successful regions resume without repeating retrieval.

These are selected-pixel summaries, not full-field averages. A rise in an index does not by itself establish planting or crop identity.


### 7.1 Calculate composite curves and plots

Use the scaled pixel bands and quality records from Step 6. The first plot is displayed as an example.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Composite timeline rows: <number>` | One summary for each region/half-month bin, including ineligible bins. This is not the number of valid observations. |
| `Vegetation timeline plot` | The upper panel summarizes selected-pixel NDVI/NDRE/NDMI; the lower panel shows valid-pixel coverage. Composite midpoints are display dates, not acquisition dates. |
<!-- output-guide:end -->


In [ ]:
timeline_folder = analysis_folder / "timelines"
composite_rows = crop_workflow.composite_timelines(quality_rows, pixel_rows, timeline_folder)
crop_workflow.plot_timelines(
    composite_rows,
    timeline_folder / "composite_plots",
    title="Semi-monthly composite summaries (bin midpoints)",
)
print("Composite timeline rows:", len(composite_rows))

# Optional visual preview of one saved plot.
from IPython.display import Image, display

plot_files = sorted((timeline_folder / "composite_plots").glob("*.png"))
if plot_files:
    display(Image(filename=str(plot_files[0])))


### Saved daily summaries

Actual-date summaries are retrieved in the two source notebooks, combined during Step 6.1, and loaded automatically by Step 8.1. Composite curves from Step 7.1 remain available for comparison. No connection or acquisition helper variables are needed here.


## 8. Identify candidate green-up intervals

The detector looks for a supported rise after a low-vegetation period. It uses actual daily summaries where available and falls back to composites for other regions. Composite-only candidates remain provisional.

| Development setting | Default |
|---|---:|
| Relative rise above baseline | 20% of the estimated amplitude |
| Minimum NDVI amplitude | 0.15 |
| Maximum gap around onset | 30 days |
| Later confirmations | 2 within 30 days |
| Baseline lookback | 120 days |
| Minimum valid sample fraction | 80% |

These are proposed settings, not validated vegetable constants. Baseline uses a local 10th percentile before each candidate peak. Short-neighbour median smoothing does not interpolate long gaps. Raw observations must also support the rise and confirmation.

Report an uncertainty interval rather than an exact planting date. Flag weak signals, gaps, multiple possible cycles and intervals crossing month boundaries. Step 9 must validate the detector and annual-label relevance before accepting any training season. Cycles near year boundaries may require additional months of observations.


### 8.1 Run the candidate detector

Prefer observed rows and preserve composite-only fallbacks. Save outcomes and the exact development settings.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Detector outcomes: {<status>: <number>}` | Counts candidate-cycle outcomes: candidate is a supported actual-date rise; provisional is composite-only; ambiguous has competing cycles; rejected fails the detection rules. One region can produce several outcomes. |
| `Results: <path>` | Where candidate intervals and detector settings are saved. These are timing proposals, not approved crop seasons or four-month feature tables. |
<!-- output-guide:end -->


In [ ]:
detector_settings = dict(crop_workflow.DEFAULT_DETECTOR)
# Edit these visible development settings before running sensitivity checks.
detector_settings["relative_threshold"] = 0.20

observed_path = timeline_folder / "observed_timelines.json"
observed_rows = (
    json.loads(observed_path.read_text(encoding="utf-8")) if observed_path.exists() else []
)
observed_regions = {row["region_id"] for row in observed_rows}
detector_rows = observed_rows + [
    row for row in composite_rows if row["region_id"] not in observed_regions
]
greenup_results = crop_workflow.detect_greenup(detector_rows, detector_settings)
greenup_folder = analysis_folder / "greenup"
crop_workflow.save_json(greenup_folder / "candidate_greenup.json", greenup_results)
crop_workflow.save_json(
    greenup_folder / "detector_settings.json",
    {
        "parameters": detector_settings,
        "validated": False,
        "label_relevance_verified": False,
    },
)
columns = [
    "region_id",
    "partition_group",
    "status",
    "reason",
    "source_kind",
    "onset_interval_start",
    "onset_interval_end",
    "peak_date",
    "baseline",
    "amplitude",
    "ndvi_threshold",
    "gap_days",
    "confirmations",
    "month_boundary_uncertain",
    "label_relevance",
    "validated",
]
crop_workflow.write_table(greenup_folder / "candidate_greenup.csv", greenup_results, columns)
print("Detector outcomes:", dict(Counter(row["status"] for row in greenup_results)))
print("Results:", greenup_folder)


### 8.2 Compare development thresholds

Repeat detection at 15%, 20% and 25% relative amplitude without using final test labels.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `No printed output` | Saves threshold_sensitivity.json with detector reruns at 15%, 20% and 25% amplitude. These are robustness checks on the same data, not independent accuracy measurements. |
<!-- output-guide:end -->


In [ ]:
# Compare development thresholds without using final test labels.
sensitivity = {}
for fraction in [0.15, 0.20, 0.25]:
    results = crop_workflow.detect_greenup(
        detector_rows, {**detector_settings, "relative_threshold": fraction}
    )
    sensitivity[str(fraction)] = results
crop_workflow.save_json(greenup_folder / "threshold_sensitivity.json", sensitivity)


### 8.3 Plot candidate uncertainty intervals

Plot the primary detector output. Shaded intervals are estimates, not verified planting dates.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `No printed output; saved green-up plots` | Writes plots to the greenup/plots folder. Shaded bands show onset uncertainty, not exact planting dates; no plot is displayed automatically here. |
<!-- output-guide:end -->


In [ ]:
crop_workflow.plot_greenup(detector_rows, greenup_results, greenup_folder / "plots")


### Outputs and interpretation

`analysis/2020/greenup/` contains candidate intervals, detector settings, sensitivity results and plots. Candidate, provisional, ambiguous and rejected outcomes remain separate. No season is automatically accepted for training.


## 9. Automatically assess timing and separate window outcomes

**No manual curve review is required for exploratory timing selection.** The code first proposes single-cycle, actual-date windows, then runs a predefined automatic quality policy.

| Check | Default rule | Purpose |
|---|---|---|
| Actual onset support | Valid observations bracket the NDVI threshold | Avoid accepting a fitted rise without real support |
| Onset gap | At most 20 days; endpoints in one month | Bound timing uncertainty |
| Nearby baseline | At least 2 observations in the preceding 30 days | Support the low-vegetation baseline |
| Sustained growth | At least 2 later confirmations within 30 days; at least 75% of post-onset observations above threshold | Reject isolated spikes |
| Confirmation gaps | At most 20 days between supporting observations | Avoid calling sparse observations continuous growth |
| NDRE agreement | Post-onset median exceeds baseline median by at least 0.02 | Check chlorophyll-related evidence alongside NDVI |
| Sensitivity | Rerun at 15%, 20% and 25% amplitude; require one supported cycle, the same onset month and peak within 15 days | Reject timing choices sensitive to small parameter changes |

NDMI change is reported as a diagnostic; it is not required to rise because water conditions can vary. These thresholds are explicit **development choices**, not validated constants for carrot/potato. Parameter stability measures internal robustness, not independent accuracy.

**Outputs:** reference-supported seasons remain separate. Automatically passing estimates have `timing_review_status: automated_pass`, `timing_validation: algorithmic_consistency_only` and `label_relevance: unverified`. Failed proposals become unresolved, with specific reasons in `automatic_timing_audit.csv`.

The automatic policy and observation fingerprints are recorded. Competing cycles, uncertain months and detector-rejected signals remain excluded. No planting records, reviewers or field identities are invented.

Manual review remains an optional override using `accept_estimate`, and independent reference evidence can still support `accept_reference`. Neither is required for automatic exploratory timing. Complete field identities and independent local test data remain separate requirements. See [automatic timing rules](docs/methodology.md#automatic-timing-assessment).

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Reference-supported field-seasons: <number>` | Crop cycles supported by specific label/cycle evidence and frozen rules. Zero means none passed that evidence path, not that satellite growth was absent. |
| `Initial estimated proposals: <number>` | Single-cycle, actual-date proposals with an onset inside one month, before automatic quality assessment. This is a subset of all detector outcomes. |
| `Automatically qualified estimates: <number>` | Proposals passing all predefined observation, persistence, NDRE and threshold-stability checks. They may enter the exploratory path without manual curve approval, but crop-cycle relevance remains unverified. |
| `Total qualified estimated windows: <number>` | Automatic passes plus any explicitly supplied human timing overrides. Automatic passes are included in this total; do not add the two counts together. |
| `Unresolved outcomes: <number>` | Detector or automatic-assessment outcomes that could not be selected. This counts outcomes, not unique parcels; see the audit reasons. |
| `Automatic timing failures: <number>` | Initial proposals that failed at least one automatic check. These are distinct from detections rejected before the automatic assessment stage. An explicit human override, if present, is retained separately. |
| `Review templates: <path>` | Where optional human timing/reference-review templates and the field-registry template are written. A template is not a completed review or identity assignment. |
<!-- output-guide:end -->


In [ ]:
import season_workflow
import timing_quality

importlib.reload(season_workflow)
importlib.reload(timing_quality)
season_folder = analysis_folder / "seasons"
experiment_artifacts = []
review_folder = season_folder / "review"
season_workflow.prepare_review_files(greenup_results, regions, detector_settings, review_folder)
configuration_path = review_folder / "frozen_configuration.json"
reviews_path = review_folder / "season_reviews.csv"
registry_path = review_folder / "field_registry.csv"
configuration = json.loads(
    (
        configuration_path
        if configuration_path.exists()
        else review_folder / "frozen_configuration.template.json"
    ).read_text(encoding="utf-8")
)
if configuration["detector_parameters"] != detector_settings:
    raise ValueError(
        "The review configuration uses different detector settings. Review or rerun detection before selecting windows."
    )
reviews = season_workflow.read_table(reviews_path) if reviews_path.exists() else []
configuration["automatic_timing_policy"] = {
    **timing_quality.DEFAULT_TIMING_POLICY,
    **configuration.get("automatic_timing_policy", {}),
}
accepted_seasons, estimated_proposals, season_audit = season_workflow.select_season_windows(
    greenup_results, reviews, regions, configuration
)
estimated_windows, automatic_timing_rows, season_audit = season_workflow.qualify_estimated_windows(
    estimated_proposals, season_audit, detector_rows, configuration
)
season_workflow.write_records(season_folder / "automatic_timing_audit.csv", automatic_timing_rows)
season_workflow.save_json(
    season_folder / "automatic_timing_policy.json",
    {
        "quality_policy": configuration["automatic_timing_policy"],
        "detector_parameters": detector_settings,
        "independently_validated": False,
    },
)
season_workflow.write_records(season_folder / "season_review_audit.csv", season_audit)
print("Reference-supported field-seasons:", len(accepted_seasons))
print("Initial estimated proposals:", len(estimated_proposals))
print(
    "Automatically qualified estimates:",
    sum(row["timing_review_status"] == "automated_pass" for row in estimated_windows),
)
print("Total qualified estimated windows:", len(estimated_windows))
print("Unresolved outcomes:", sum(row["window_status"] == "unresolved" for row in season_audit))
print(
    "Automatic timing failures:",
    sum(row["automatic_timing_status"] == "failed" for row in automatic_timing_rows),
)
print("Review templates:", review_folder)


## 10. Save the four-calendar-month windows separately

**Rule:** start on the first day of the supported green-up month and include the following three calendar months. For example, April 8–14 green-up gives April 1 through August 1 (exclusive): April–July.

Save reference-supported seasons and satellite-estimated windows to different files. Each record retains its onset interval, source kind, timing-review status, crop-label relevance and selection reason. These are observation windows, not guaranteed planting-to-harvest durations.

Month-boundary uncertainty is not resolved automatically. Windows can cross years; absent next-year imagery will remain missing in the feature/loss reports. Settings can remain draft while you inspect estimated windows. Freezing is still required before training.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Reference-supported windows saved: <number>` | Rows saved to accepted_seasons.csv/.json for the primary path. Zero can be expected when no independent crop-cycle evidence was supplied. |
| `Estimated observation windows saved: <number>` | Qualified four-calendar-month windows saved separately to estimated_windows.csv/.json. These are estimated observation periods, not verified planting-to-harvest durations. |
| `Audit: <path>` | The full selection/exclusion log. Initial proposals and automatic checks are also retained in separate files, including proposals that did not qualify. |
<!-- output-guide:end -->


In [ ]:
season_workflow.write_records(season_folder / "estimated_proposals.csv", estimated_proposals)
season_workflow.save_json(season_folder / "estimated_proposals.json", estimated_proposals)
season_workflow.write_records(season_folder / "accepted_seasons.csv", accepted_seasons)
season_workflow.save_json(season_folder / "accepted_seasons.json", accepted_seasons)
season_workflow.write_records(season_folder / "estimated_windows.csv", estimated_windows)
season_workflow.save_json(season_folder / "estimated_windows.json", estimated_windows)
print("Reference-supported windows saved:", len(accepted_seasons))
print("Estimated observation windows saved:", len(estimated_windows))
print("Audit:", season_folder / "season_review_audit.csv")


## 11. Select eight composites and derive separate feature tables

Use the scaled selected-pixel bands and quality records from Step 6. Select the eight calendar half-month bins in each window and derive NDVI, NDRE and NDMI directly from the median bands. B2 remains an inspection band.

**Two outputs:** `features/complete_features.csv` contains only reference-supported seasons. `estimated_features/complete_features.csv` contains satellite-estimated windows, with automated assessment evidence retained. Estimates do not silently enter the primary table.

Each complete pixel has 64 features; the matching 40 band features remain available. No second temporal median or gap interpolation is performed.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Reference-supported complete pixel windows: <number>` | Pixel/window rows with every required feature present, drawn only from reference-supported seasons. Zero can mean there were no reference-supported input seasons. |
| `Estimated complete pixel windows: <number>` | Pixel/window rows from qualified estimated windows that pass all eight bins and contain all 64 features. These are complete feature rows, not independent fields or already partitioned model inputs. |
| `Features per complete pixel: <number>` | Model inputs per pixel/window: 8 half-month bins × (5 bands + 3 indices) = 64. Coordinates, counts and review/evidence fields are metadata, not additional model features. |
<!-- output-guide:end -->


**Final pilot selection:** retain at most 1,000 unique external pixel locations per crop across reference-supported and estimated windows. Prefer reference-supported windows when a location appears twice, and draw round-robin across parcel groups. Local registry rows are preserved. Selection uses no model scores or test accuracy.

Output: `Usable selection by crop: <records>` — available unique pixels, retained pixels and shortfall for each crop. Counts below 1,000 are reported without filling gaps.

`all_complete_features.csv` preserves every quality-passing window; `complete_features.csv` contains the selected pilot. Quality-loss reports describe the pre-trim pool; budget trimming is not a cloud-quality exclusion.


In [ ]:
feature_rows, feature_exclusions, sample_losses, bin_coverage = season_workflow.build_features(
    accepted_seasons,
    pixel_rows,
    quality_rows,
    minimum_fraction=configuration["minimum_valid_fraction"],
)
estimated_feature_rows, estimated_exclusions, estimated_losses, estimated_bin_coverage = (
    season_workflow.build_features(
        estimated_windows,
        pixel_rows,
        quality_rows,
        minimum_fraction=configuration["minimum_valid_fraction"],
    )
)
import pilot_sampling

feature_folder = season_folder / "features"
estimated_feature_folder = season_folder / "estimated_features"
# Keep the full quality-passing pool so budget exclusions remain distinguishable from missing data.
season_workflow.write_records(feature_folder / "all_complete_features.csv", feature_rows)
season_workflow.write_records(
    estimated_feature_folder / "all_complete_features.csv", estimated_feature_rows
)
pilot_registry = season_workflow.read_table(registry_path) if registry_path.exists() else []
feature_rows, estimated_feature_rows, usable_selection_summary = pilot_sampling.trim_external_pilot(
    feature_rows,
    estimated_feature_rows,
    pilot_registry,
    target_per_crop=1000,
    seed=configuration["random_seed"],
    crops=("carrot", "cabbage", "potato"),
)
season_workflow.write_records(
    season_folder / "usable_pixel_selection.csv", usable_selection_summary
)
print("Usable selection by crop:", usable_selection_summary)
season_workflow.write_records(feature_folder / "complete_features.csv", feature_rows)
season_workflow.write_records(
    estimated_feature_folder / "complete_features.csv", estimated_feature_rows
)
print("Reference-supported complete pixel windows:", len(feature_rows))
print("Estimated complete pixel windows:", len(estimated_feature_rows))
print("Features per complete pixel:", len(season_workflow.feature_columns()))


## 12. Apply the complete-data policy and inspect both sets of losses

Require all 64 finite features and passing composite quality in all eight bins. Nodata, zero index denominators, missing year-boundary imagery and failed quality cause exclusion. No interpolation is applied.

Both `features/` and `estimated_features/` contain `sample_losses.csv`, `bin_coverage.csv` and `feature_exclusions.csv`. Inspect them by crop, region, group and window status. Selected pixels with no readable records remain counted.

A valid timing estimate can still lose every pixel to incomplete imagery. That is a feature-coverage failure, separate from an unresolved onset. Both stages remain visible in the reports.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Reference-supported excluded pixel windows: <number>` | Selected reference-supported pixel/windows that fail the complete-data policy. Zero may mean no reference-supported windows were available to assess. |
| `Estimated excluded pixel windows: <number>` | Selected pixels in qualified estimated windows that fail at least one required bin or feature. Missing next-year imagery, low regional coverage and invalid indices can all cause exclusion. This count excludes proposals already rejected at the timing stage. |
<!-- output-guide:end -->


In [ ]:
season_workflow.write_records(feature_folder / "feature_exclusions.csv", feature_exclusions)
season_workflow.write_records(feature_folder / "sample_losses.csv", sample_losses)
season_workflow.write_records(feature_folder / "bin_coverage.csv", bin_coverage)
season_workflow.write_records(
    estimated_feature_folder / "feature_exclusions.csv", estimated_exclusions
)
season_workflow.write_records(estimated_feature_folder / "sample_losses.csv", estimated_losses)
season_workflow.write_records(estimated_feature_folder / "bin_coverage.csv", estimated_bin_coverage)
print(
    "Reference-supported excluded pixel windows:",
    sum(row["excluded_pixels"] for row in sample_losses),
)
print("Estimated excluded pixel windows:", sum(row["excluded_pixels"] for row in estimated_losses))


## 13. Assign fields and keep the exploratory experiment separate

Complete `field_registry.csv` with reviewed independent identities, source (`external` or `local`) and partition (`train`, `validation` or `test`). All fragments, seasons and pixels of one field stay together. Coordinate duplicates and conflicting identities stop execution.

**Primary rows** use reference-supported windows. **Exploratory rows** add only timing-qualified estimates assigned to external training fields. Estimated windows cannot supply local training labels or independent test truth. Final test rows require confirmed labels and reference-supported windows.

If no registry is available, the window and feature exports remain inspectable, but partitioned inputs stay pending. Failed automatic proposals remain in `estimated_proposals.csv` and the audit; only qualified estimates become model inputs.

To add local data, process and review it using the same schema, then combine its reference-supported feature rows with `feature_rows` before this cell. Include its inputs and losses in Step 14 provenance. See [the local-data instructions](docs/training.md#local-data-and-field-partitions).

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Field assignment pending: complete field_registry.csv before preparing model inputs.` | Complete feature rows exist or can be inspected, but no completed field registry is available. Timing qualification does not establish independent field identities. |
| `Primary partitioned pixel windows: <number>` | Reference-supported complete rows assigned to train/validation/test using the field registry. Zero means the primary model-input table is empty; inspect upstream counts and registry availability. |
| `Exploratory partitioned pixel windows: <number>` | Reference rows plus timing-qualified estimates assigned only to external training fields. Zero does not mean complete estimated features were deleted; field assignments or eligible inputs may still be missing. |
<!-- output-guide:end -->


In [ ]:
field_registry = season_workflow.read_table(registry_path) if registry_path.exists() else []
partitioned_rows, exploratory_rows = [], []
reviewed_estimated_features = [
    row for row in estimated_feature_rows if season_workflow.timing_is_qualified(row)
]
if field_registry:
    partitioned_rows = season_workflow.assign_partitions(feature_rows, field_registry)
    if reviewed_estimated_features:
        exploratory_rows = season_workflow.prepare_exploratory_rows(
            partitioned_rows, reviewed_estimated_features, field_registry
        )
else:
    print("Field assignment pending: complete field_registry.csv before preparing model inputs.")
season_workflow.write_records(season_folder / "partitioned_features.csv", partitioned_rows)
season_workflow.write_records(
    season_folder / "sample_counts.csv", season_workflow.sample_counts(partitioned_rows)
)
season_workflow.write_records(
    season_folder / "exploratory_partitioned_features.csv", exploratory_rows
)
print("Primary partitioned pixel windows:", len(partitioned_rows))
print("Exploratory partitioned pixel windows:", len(exploratory_rows))


## Optional experiment: Coverage threshold comparison

Compare 70%, 80% and 90% clear-pixel coverage using saved actual-date summaries (or composite summaries when no daily records exist). Outputs preserve original quality flags, counts and a separate experimental acceptance flag.

This tests the **sample-region coverage threshold**. It does not change the cloud-probability masks or independently measure residual clouds. Inspect development imagery before freezing a threshold; do not tune using test fields.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Coverage comparison: <path>` | Folder with per-date decisions and counts for 70%, 80% and 90% coverage on assigned development regions. It is empty when no eligible development regions are registered; it does not change the cloud mask or the primary threshold. |
<!-- output-guide:end -->


In [ ]:
coverage_experiment_folder = season_folder / "experiments" / "coverage"
development_regions = season_workflow.development_region_ids(regions, field_registry)
development_timelines = [row for row in detector_rows if row["region_id"] in development_regions]
coverage_decisions, coverage_summary_rows = season_workflow.summarize_coverage_thresholds(
    development_timelines,
    thresholds=(0.7, 0.8, 0.9),
)
season_workflow.write_records(coverage_experiment_folder / "decisions.csv", coverage_decisions)
season_workflow.write_records(coverage_experiment_folder / "summary.csv", coverage_summary_rows)
print("Coverage comparison:", coverage_experiment_folder)
experiment_artifacts.extend(
    [coverage_experiment_folder / "decisions.csv", coverage_experiment_folder / "summary.csv"]
)


## Optional experiment: Window sensitivity analysis

Build eight consecutive half-month bins beginning one bin earlier, at the primary start, or one bin later. Shifted variants are labeled explicitly; they are not four whole calendar months. Preserve primary dates and missing-data losses for comparison.

Run this on development seasons only. Missing year-boundary bins remain missing. Compare coverage before considering a separately predefined model experiment; the detector evidence and accepted crop labels are unchanged.

The development comparison includes both reference-supported and estimated windows, retaining their distinct status metadata.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Window comparison: <path>` | Folder with feature/loss/coverage reports for windows shifted one half-month earlier, unchanged, or later. Only registered development regions are used; empty reports can mean no development identities are assigned. |
<!-- output-guide:end -->


In [ ]:
window_experiment_folder = season_folder / "experiments" / "windows"
development_regions = season_workflow.development_region_ids(regions, field_registry)
development_seasons = [
    season
    for season in accepted_seasons + estimated_windows
    if season["region_id"] in development_regions
]
for bin_shift in (-1, 0, 1):
    variant_folder = window_experiment_folder / f"shift_{bin_shift:+d}"
    shifted_features, shifted_exclusions, shifted_losses, shifted_coverage = (
        season_workflow.build_features(
            development_seasons,
            pixel_rows,
            quality_rows,
            minimum_fraction=configuration["minimum_valid_fraction"],
            bin_shift=bin_shift,
        )
    )
    season_workflow.write_records(variant_folder / "features.csv", shifted_features)
    season_workflow.write_records(variant_folder / "exclusions.csv", shifted_exclusions)
    season_workflow.write_records(variant_folder / "losses.csv", shifted_losses)
    season_workflow.write_records(variant_folder / "coverage.csv", shifted_coverage)
    experiment_artifacts.extend(
        [
            variant_folder / filename
            for filename in ("features.csv", "exclusions.csv", "losses.csv", "coverage.csv")
        ]
    )
print("Window comparison:", window_experiment_folder)


## 14. Export primary and exploratory training inputs

Export two distinct portable bundles, each with feature rows, configuration, coverage and file hashes:

| Folder | Policy |
|---|---|
| `training_input/` | Reference-supported windows only |
| `exploratory_training_input/` | The same reference rows plus timing-qualified external training estimates |

Estimated proposals and their features can be saved even when settings are not frozen or field identities are missing. Such proposals are not automatically training-ready. Both experiments still require frozen development settings, reviewed field partitions, complete features, class coverage and independent local test data before fitting.

The training notebook defaults to the primary bundle. Point it explicitly to `exploratory_training_input/training_handoff.json` for the estimated-window experiment; its models are written to a separate output folder.

Include any additional local input paths and loss records in the provenance below. Preserve all bundle files together when moving PCs.

<!-- output-guide:start -->
**How to read the output**

The placeholders `<number>`, `<value>` and `<path>` stand for the actual values printed when you run the next cell.

| Output shown | Meaning and purpose |
|---|---|
| `Primary training handoff: <path>` | Manifest for the reference-supported training-input bundle. A saved manifest can describe an empty pending bundle; its existence does not establish training readiness. |
| `Exploratory training handoff: <path>` | Separate manifest for reference rows plus qualified external training estimates. The training notebook must explicitly load this bundle for the exploratory experiment. |
| `Preparation status: prepared or pending_inputs` | This displayed status summarizes whether the primary table has partitioned rows. It does not summarize exploratory readiness or prove that training classes, frozen rules and local test data are complete; each bundle records its own status and the training notebook checks prerequisites. |
| `Next: open Crop_Model_Training.ipynb and run its settings cell.` | A navigation instruction. Open the bundle for inspection, but fitting still requires frozen development settings, class coverage and independent local test data. |
<!-- output-guide:end -->


In [ ]:
coverage_summary = {
    "usable_pixel_selection": usable_selection_summary,
    "source_manifests": [str(path) for path in source_manifest_paths],
    "detector_regions": len({row["region_id"] for row in greenup_results}),
    "detector_outcomes": dict(Counter(row["status"] for row in greenup_results)),
    "review_outcomes": dict(Counter(row["decision"] for row in season_audit)),
    "accepted_seasons": len(accepted_seasons),
    "estimated_windows": len(estimated_windows),
    "estimated_proposals": len(estimated_proposals),
    "automatic_timing_outcomes": dict(
        Counter(row["automatic_timing_status"] for row in automatic_timing_rows)
    ),
    "estimated_timing_reviews": dict(
        Counter(row["timing_review_status"] for row in estimated_windows)
    ),
    "unresolved_outcomes": sum(row["window_status"] == "unresolved" for row in season_audit),
    "estimated_complete_pixel_windows": len(estimated_feature_rows),
    "estimated_sample_losses": estimated_losses,
    "exploratory_sample_counts": season_workflow.sample_counts(exploratory_rows),
    "complete_pixel_windows": len(feature_rows),
    "partitioned_pixel_windows": len(partitioned_rows),
    "excluded_pixel_windows": sum(row["excluded_pixels"] for row in sample_losses),
    "sample_losses": sample_losses,
    "sample_counts": season_workflow.sample_counts(partitioned_rows),
    "preparation_status": "prepared" if partitioned_rows else "pending_inputs",
    "mapping_coverage": "not_measured",
}
training_input_folder = season_folder / "training_input"
reference_configuration = {**configuration, "window_policy": "reference_supported_only"}
handoff_path = season_workflow.export_training_handoff(
    training_input_folder, partitioned_rows, reference_configuration, coverage_summary
)
exploratory_configuration = {**configuration, "window_policy": "exploratory_external_estimates"}
exploratory_input_folder = season_folder / "exploratory_training_input"
exploratory_handoff_path = season_workflow.export_training_handoff(
    exploratory_input_folder, exploratory_rows, exploratory_configuration, coverage_summary
)
report_artifacts = [
    season_folder / "usable_pixel_selection.csv",
    feature_folder / "all_complete_features.csv",
    estimated_feature_folder / "all_complete_features.csv",
    season_folder / "season_review_audit.csv",
    season_folder / "accepted_seasons.csv",
    season_folder / "accepted_seasons.json",
    season_folder / "estimated_windows.csv",
    season_folder / "estimated_windows.json",
    season_folder / "estimated_proposals.csv",
    season_folder / "estimated_proposals.json",
    season_folder / "automatic_timing_audit.csv",
    season_folder / "automatic_timing_policy.json",
    season_folder / "exploratory_partitioned_features.csv",
    estimated_feature_folder / "complete_features.csv",
    estimated_feature_folder / "feature_exclusions.csv",
    estimated_feature_folder / "sample_losses.csv",
    estimated_feature_folder / "bin_coverage.csv",
    exploratory_handoff_path,
    exploratory_input_folder / "partitioned_features.csv",
    exploratory_input_folder / "configuration.json",
    exploratory_input_folder / "coverage.json",
    feature_folder / "complete_features.csv",
    feature_folder / "feature_exclusions.csv",
    feature_folder / "sample_losses.csv",
    feature_folder / "bin_coverage.csv",
    season_folder / "partitioned_features.csv",
    season_folder / "sample_counts.csv",
    handoff_path,
    training_input_folder / "partitioned_features.csv",
    training_input_folder / "configuration.json",
    training_input_folder / "coverage.json",
]
preprocessing_manifest = season_workflow.package_run(
    season_folder / "preprocessing_package",
    configuration,
    input_paths=[
        configuration_path,
        reviews_path,
        registry_path,
        greenup_folder / "candidate_greenup.json",
        greenup_folder / "detector_settings.json",
        analysis_folder / "quality" / "selected_pixel_bands.csv",
        analysis_folder / "quality" / "composite_quality.csv",
        analysis_folder / "source_inputs.json",
        Path("pilot_sampling.py"),
        Path("pilot_sources.py"),
        Path("francecrops_workflow.py"),
        Path("satellite_workflow.py"),
        Path("FranceCrops_Satellite_Preparation.ipynb"),
        Path("crop_workflow.py"),
        Path("season_workflow.py"),
        Path("timing_quality.py"),
        Path("requirements-notebooks.txt"),
        Path("Sen4AgriNet_Satellite_Preparation.ipynb"),
        Path("Crop_Four_Month_Windows.ipynb"),
    ],
    artifact_paths=report_artifacts + experiment_artifacts,
    coverage=coverage_summary,
)
print("Primary training handoff:", handoff_path)
print("Exploratory training handoff:", exploratory_handoff_path)
print("Preparation status:", coverage_summary["preparation_status"])
print("Next: open Crop_Model_Training.ipynb and run its settings cell.")


## Research and documentation

- [S4A downloads](https://huggingface.co/datasets/orion-ai-lab/S4A)
- [S4A repository and taxonomy](https://github.com/Orion-AI-Lab/S4A)
- [S4A paper](https://arxiv.org/abs/2204.00951)
- [L2A collection](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)
- [Copernicus start of season: PPI amplitude method](https://www.copernicus.eu/en/access-data/copernicus-services-catalogue/start-season-date-2017-present-raster-10-m-europe-yearly)
- [Carrot Sentinel-2 development study](https://link.springer.com/article/10.1007/s11119-023-10083-z)

Literature supports methods, not field-specific planting/harvest dates. Proposed detector parameters require validation.

## Added research references

- [Crop-mapping review](https://doi.org/10.1080/10106049.2026.2690363)
- [Feature optimization](https://www.sciencedirect.com/science/article/pii/S1569843224005284)
- [Cloud quality and cross-year yield evaluation](https://ira.agroscope.ch/de-CH/Page/Einzelpublikation/Download?einzelpublikationId=56073)

These support experimental ideas; they do not validate the exact four-month design or crop-specific onset thresholds.


## Preprocessing summary and limitations

This notebook prepares reviewed four-month windows, complete features, independent-field partitions and a portable training-input bundle. It exports reference-supported and satellite-estimated windows separately and does not fit or evaluate models.

Candidate green-up is not a verified planting date. Annual crop declarations need cycle-specific review. Unresolved cycles remain excluded; initial proposals and automatic assessments are preserved separately, with passing estimates available for exploratory features. Sampling-mask pixels are correlated within fields, and the expanded pilot targets carrot, cabbage and potato, using source-specific observation years.

Continue with [Crop_Model_Training.ipynb](Crop_Model_Training.ipynb) after exporting the bundle. Training reads saved files and can use a separate kernel or PC.
